# Baseline: SMIRK_em1 under synthetic occlusions

Evaluates the pretrained SMIRK encoder on all clean test faces, clean and with synthetic occlusions
(`src/evaluation/occlusion_benchmark.py`): random parts, and exactly the mouth / eyes / eyebrows / nose.
No rendering is needed, so PyTorch3D is not installed.

Before running:
- Settings: **Accelerator GPU T4 x1**, **Internet on**; Secrets `FLAME_USERNAME`, `FLAME_PASSWORD` enabled.
- Inputs: `clean-faces` (+ `clean-faces-rafdb` if available) and the matching image datasets
  `expw-crops`, `affectnet-crops` (+ `rafdb-crops`).

Output `/kaggle/working/baseline_smirk_em1/`: `summary.csv`, `per_image.csv`, `features.npz`, `config.json`.
Keep the notebook and outputs **Private** (FLAME / SMIRK licenses).

## 1. Environment and code

In [ ]:
import sys, os, glob, subprocess
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| GPU', torch.cuda.is_available())
assert torch.cuda.is_available(), 'enable the GPU accelerator'

os.chdir('/kaggle/working')  # leave /tmp/smirk before deleting it (a later cell changes into it)
!rm -rf /tmp/smirk && git clone -q --depth 1 -b occlusion-aware https://github.com/THUONGVIP09/smirk.git /tmp/smirk
!cd /tmp/smirk && git log --oneline -1
!pip install -q timm==0.9.16 gdown

## 2. FLAME 2020 and SMIRK_em1.pt

In [ ]:
import shutil
from kaggle_secrets import UserSecretsClient
from urllib.parse import quote

assert os.path.isdir('/tmp/smirk/assets'), 'the SMIRK code is missing in /tmp/smirk: run the cell that clones the repository first'
os.makedirs('/tmp/smirk/assets/FLAME2020', exist_ok=True)

secrets = UserSecretsClient()
post_data = 'username=' + quote(secrets.get_secret('FLAME_USERNAME')) + '&password=' + quote(secrets.get_secret('FLAME_PASSWORD'))
subprocess.run(['wget', '-q', '--post-data', post_data, '--no-check-certificate', '-O', '/tmp/FLAME2020.zip',
                'https://download.is.tue.mpg.de/download.php?domain=flame&sfile=FLAME2020.zip&resume=1'], check=True)
del post_data
assert subprocess.run(['unzip', '-q', '-o', '/tmp/FLAME2020.zip', '-d', '/tmp/smirk/assets/FLAME2020/']).returncode == 0, \
    'the download is not a zip file: check the FLAME_USERNAME / FLAME_PASSWORD secrets'
found = glob.glob('/tmp/smirk/assets/FLAME2020/**/generic_model.pkl', recursive=True)
assert found, 'generic_model.pkl is not in the FLAME zip'
if os.path.dirname(found[0]) != '/tmp/smirk/assets/FLAME2020':  # the zip may contain a FLAME2020/ folder
    for name in os.listdir(os.path.dirname(found[0])):
        shutil.move(os.path.join(os.path.dirname(found[0]), name), '/tmp/smirk/assets/FLAME2020/')

!mkdir -p /tmp/smirk/pretrained_models
!gdown -q 1T65uEd9dVLHgVw5KiUYL66NUee-MCzoE -O /tmp/smirk/pretrained_models/SMIRK_em1.pt
!ls -la /tmp/smirk/assets/FLAME2020/generic_model.pkl /tmp/smirk/pretrained_models/SMIRK_em1.pt

## 3. Find the inputs

In [ ]:
CLEAN_DIRS = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/clean_meta.csv', recursive=True))
IMAGE_DIRS = {}
for folder in glob.glob('/kaggle/input/**/images', recursive=True):
    if not os.path.isdir(folder):
        continue
    # dataset of an image folder, from the keyword in its path (works for rafdb_crops, rafdb-crops, affectnet_1, ...)
    path = os.path.relpath(folder, '/kaggle/input').lower()
    name = next((k for k in ('expw', 'affectnet', 'rafdb') if k in path), None)
    if name:
        IMAGE_DIRS[name] = folder
        print(f'{name}: {folder} ({len(os.listdir(folder))} images)')
print('clean dirs:', CLEAN_DIRS)
print('image dirs:', IMAGE_DIRS)
if not IMAGE_DIRS:
    !find /kaggle/input -maxdepth 6 -type d | head -40
assert CLEAN_DIRS and IMAGE_DIRS, 'add clean-faces and the *-crops datasets as input'

ARGS = ' '.join([f'--clean_dir "{d}"' for d in CLEAN_DIRS] + [f'--image_dir "{k}={v}"' for k, v in IMAGE_DIRS.items()])
os.chdir('/tmp/smirk')  # FLAME assets are loaded with paths relative to the repository root

## 4. Quick check on 200 test faces (a few minutes)

In [ ]:
!python -m src.evaluation.occlusion_benchmark {ARGS} --checkpoint pretrained_models/SMIRK_em1.pt --out_dir /tmp/quick_check --limit 200 --probe_train_max 3000

## 5. Full baseline on all test faces

In [ ]:
OUT = '/kaggle/working/baseline_smirk_em1'
!python -m src.evaluation.occlusion_benchmark {ARGS} --checkpoint pretrained_models/SMIRK_em1.pt --out_dir {OUT}

## 6. Results

`expr_mm`: how far (mm, mean over the FLAME vertices) the occluded-image expression mesh is from the clean one.
`exp_scale`: ||psi_occluded|| / ||psi_clean||, > 1 = more intense expressions under occlusion (SMIRK paper, Sec. 4.5).
`emotion_mean_class_acc`: MLP emotion probe, accuracy averaged over the classes (8 for AffectNet, 7 for ExpW / RAF-DB).
The SMIRK paper reports 0.64 (8 classes) on the official AffectNet test set; our AffectNet split differs.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

summary = pd.read_csv(f'{OUT}/summary.csv')
pd.set_option('display.width', 200)
print(summary.round(3).to_string(index=False))

overall = summary[summary['dataset'] == 'all'].set_index('protocol')
BLUE, INK, MUTED = '#2a78d6', '#0b0b0b', '#52514e'

def bars(ax, values, title, fmt):
    ax.bar(values.index, values.values, color=BLUE, width=0.6)
    for x, v in zip(values.index, values.values):
        ax.text(x, v, fmt.format(v), ha='center', va='bottom', fontsize=9, color=INK)
    ax.set_title(title, loc='left', fontsize=11, color=INK)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(colors=MUTED)
    ax.grid(axis='y', color='#e5e5e3', linewidth=0.8); ax.set_axisbelow(True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
bars(axes[0], overall['expr_mm'].drop('clean'), 'Expression error vs clean image (mm)', '{:.2f}')
bars(axes[1], overall['exp_scale'].drop('clean'), 'Expression intensity, occluded / clean', '{:.2f}')
axes[1].axhline(1, color=MUTED, linewidth=1, linestyle='--')
bars(axes[2], overall['emotion_mean_class_acc'] * 100, 'Emotion probe, mean class accuracy (%)', '{:.1f}')
plt.tight_layout(); plt.show()

## 7. Save

*Save Version → Save & Run All*, then create a **Private** dataset from the output (for example `baseline-smirk-em1`),
or just download `summary.csv`.